# 01 · Deduplicar UNA base — nivel BASICO

**Que hace**: encuentra registros repetidos de la misma empresa dentro de un solo
archivo (Excel o CSV) usando el NIT y la razon social, y le entrega la **carpeta del
estandar de salida** (`correlativa.parquet`, `excel/correlativa.xlsx`, `manifest.json`)
mas una vista `DUPLICADOS` con los grupos de duplicados.

**Que necesita**: un archivo con al menos dos columnas — una con el **NIT**
(puede venir sucio o vacio en algunas filas) y otra con el **nombre / razon social**.

**Como usarlo (3 pasos)**:
1. Ejecute la **Celda A** (prepara el entorno; solo una vez por sesion) y la
   **Celda A1** (rutas: por defecto `/content/datos` y `/content/resultados/01`).
2. En la **Celda B**, escriba el nombre de su archivo y los nombres de sus columnas.
3. Ejecute la **Celda B** y descargue la carpeta de resultados.

> En Colab Free, suba el archivo a `/content/datos` y descargue `/content/resultados`
> antes de cerrar la sesion; el disco local es efimero.
>
> Motor: `rues-linker 0.22.4` (`dedupe`, ruta canonica de produccion; el resultado
> es un `ResultadoLinkage` con el contrato de salida 1.0).
> Si necesita cruzar **dos** bases use el notebook `02`; para varias fuentes, el `03`.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CELDA A — ENTORNO  (ejecutar UNA vez por sesion; no requiere edicion)
# ══════════════════════════════════════════════════════════════════════════
from __future__ import annotations

import importlib
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

VERSION_OBJETIVO = "0.22.4"
ORIGEN_GIT = "rues-linker @ git+https://github.com/EnriqueForero/rues-linker_v2@v0.22.4"


def _version_ok(v: str, objetivo: str = VERSION_OBJETIVO) -> bool:
    """True si la version instalada coincide con el release objetivo."""

    def _t(x: str) -> tuple[int, ...]:
        base = x.split("+")[0].split("-")[0]
        return tuple(int(p) for p in base.split(".")[:3] if p.isdigit())

    try:
        return _t(v) == _t(objetivo)
    except Exception:
        return False


def preparar_entorno() -> None:
    """Garantiza exactamente rues-linker 0.22.4 desde el tag oficial."""
    try:
        instalada = version("rues-linker")
    except PackageNotFoundError:
        instalada = None
    if instalada is not None and _version_ok(instalada):
        import record_linkage

        if not _version_ok(record_linkage.__version__):
            raise RuntimeError(
                "Hay otra version cargada en memoria. Reinicie el entorno y reintente."
            )
        print(f"rues-linker {record_linkage.__version__} ya instalado — listo")
        return
    if instalada is None:
        print("rues-linker no instalado; instalando desde GitHub...")
    else:
        print(f"rues-linker {instalada} no coincide con {VERSION_OBJETIVO}; instalando...")
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", ORIGEN_GIT],
        capture_output=True,
        text=True,
    )
    if r.returncode != 0:
        raise RuntimeError(
            "No fue posible instalar rues-linker.\n"
            "  Verifique que Colab tenga salida a GitHub y vuelva a ejecutar esta celda.\n"
            f"  Detalle pip: {r.stderr.strip()[-400:]}"
        )
    importlib.invalidate_caches()
    if "record_linkage" in sys.modules:
        raise RuntimeError(
            "La libreria se actualizo, pero una version anterior sigue cargada. "
            "Reinicie el entorno y vuelva a ejecutar esta celda."
        )
    import record_linkage

    if not _version_ok(record_linkage.__version__):
        raise RuntimeError(
            f"Se instalo {record_linkage.__version__} pero se requiere {VERSION_OBJETIVO}. "
            "Reinicie el entorno (Entorno de ejecucion -> Reiniciar) y vuelva a correr esta celda."
        )
    print(f"rues-linker {record_linkage.__version__} instalado — listo")


preparar_entorno()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CELDA A1 — RUTAS  (donde estan sus datos y donde quedan los resultados)
# ══════════════════════════════════════════════════════════════════════════
# En Colab Free suba sus archivos a /content/datos (los notebooks 01-04 no
# montan Drive) y descargue RUTA_RESULTADOS antes de cerrar la sesion: el
# disco local es efimero. Los nombres de archivo se declaran en la Celda B.
# tests/test_notebooks_contrato.py sobreescribe estas rutas para correr el
# notebook sobre datos sinteticos.
from pathlib import Path

RUTA_DATOS = Path("/content/datos")  # carpeta con sus archivos
RUTA_RESULTADOS = Path("/content/resultados/01")  # carpeta de salida

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CELDA A2 — EXTRAS (definiciones; ejecutar una vez, no requiere edicion)
# ══════════════════════════════════════════════════════════════════════════

# ── Lectura y validacion de bases (fail-fast con mensajes accionables) ──────

import gc
import time
import unicodedata
from pathlib import Path

import pandas as pd

from record_linkage import dedupe, escribir_resultado, exportar_vistas, leer_resultado


def _norm(texto: str) -> str:
    """Normaliza un nombre de columna: sin tildes, mayusculas, sin espacios extremos."""
    t = unicodedata.normalize("NFKD", str(texto))
    return "".join(ch for ch in t if not unicodedata.combining(ch)).strip().upper()


def leer_base(
    ruta: str | Path,
    hoja: str | int = 0,
    fila_encabezado: int = 0,
    columnas_texto: tuple[str, ...] = (),
) -> pd.DataFrame:
    """Lee CSV/XLSX validando existencia y devolviendo columnas normalizadas.

    Args:
        ruta: ruta local al archivo .csv, .xlsx o .xls.
        hoja: nombre o indice de hoja (solo Excel).
        fila_encabezado: fila (0-index) donde estan los encabezados.
        columnas_texto: columnas a forzar como texto (p. ej. NIT, para no
            perder ceros a la izquierda ni convertir a notacion cientifica).

    Returns:
        DataFrame con encabezados normalizados (sin tildes, en mayusculas).

    Raises:
        FileNotFoundError: si la ruta no existe (con la carpeta que si existe).
        ValueError: si la extension no es soportada.
    """
    p = Path(ruta)
    if not p.exists():
        padre = next((x for x in p.parents if x.exists()), Path("/"))
        vistos = ", ".join(sorted(f.name for f in padre.iterdir())[:12]) or "(vacia)"
        raise FileNotFoundError(
            f"No existe: {p}\n  La carpeta mas cercana que SI existe es: {padre}\n"
            f"  Contiene: {vistos}\n  Corrija la ruta o el nombre del archivo (copielo exacto)."
        )
    dtype = {c: "string" for c in columnas_texto}
    ext = p.suffix.lower()
    if ext == ".csv":
        df = pd.read_csv(p, header=fila_encabezado, dtype=dtype, encoding="utf-8")
    elif ext in (".xlsx", ".xls", ".xlsm"):
        df = pd.read_excel(p, sheet_name=hoja, header=fila_encabezado, dtype=dtype)
    else:
        raise ValueError(f"Extension no soportada: '{ext}'. Use .csv o .xlsx")
    df.columns = [_norm(c) for c in df.columns]
    return df


def exigir_columnas(df: pd.DataFrame, requeridas: dict[str, str], contexto: str) -> pd.DataFrame:
    """Renombra columnas del usuario al contrato NIT/RAZON_SOCIAL/... y valida.

    Args:
        df: base leida.
        requeridas: {nombre_en_su_archivo: nombre_contrato}; los nombres del
            archivo se comparan ya normalizados (sin tildes, mayusculas).
        contexto: etiqueta para mensajes de error (p. ej. "BASE_A").

    Returns:
        DataFrame solo con las columnas del contrato, renombradas.
    """
    mapa = {_norm(k): v for k, v in requeridas.items()}
    faltan = [k for k in mapa if k not in df.columns]
    if faltan:
        raise KeyError(
            f"[{contexto}] No se hallaron las columnas {faltan}.\n"
            f"  Columnas disponibles: {list(df.columns)}\n"
            "  Corrija los nombres en la CONFIGURACION (se comparan sin tildes ni mayusculas)."
        )
    out = df[list(mapa)].rename(columns=mapa).copy()
    for c in out.columns:
        out[c] = out[c].astype("string").fillna("").str.strip()
    return out


# ── Vistas derivadas (Excel si caben; CSV comprimido si exceden) ───────────
# La carpeta del estandar la escribe la libreria (un solo escritor:
# `escribir_resultado`). Lo que el notebook agrega son VISTAS derivadas de la
# correlativa, y van en `vistas/` dentro de esa carpeta: el manifiesto no las
# lista (no son parte del contrato) y `leer_resultado` las ignora. Las escribe
# `exportar_vistas` (la funcion libre del estandar, F2.11): un .xlsx por vista
# con las formulas neutralizadas, o un .csv.gz completo si no cabe en Excel.


def tabla_duplicados(correlativa: pd.DataFrame) -> pd.DataFrame:
    """Solo los grupos con 2 o mas registros (lo accionable), ordenados por grupo."""
    tam = correlativa.groupby("ID_GRUPO")["ID_GRUPO"].transform("size")
    return (
        correlativa[tam > 1]
        .sort_values(["ID_GRUPO", "RAZON_SOCIAL"], kind="stable")
        .reset_index(drop=True)
    )


def deduplicar_una_base(
    ruta_archivo: str | Path,
    hoja: str | int,
    col_nit: str,
    col_nombre: str,
    modo: str,
    carpeta_salida: str | Path,
) -> Path:
    """Flujo completo del nivel basico: leer -> validar -> dedupe() -> carpeta del estandar.

    Returns:
        La carpeta escrita: ``<carpeta_salida>/<AAAA-MM-DD_HHMM>_deduplicacion/`` con
        ``correlativa.parquet``, ``excel/correlativa.xlsx``, ``diccionario.csv``,
        ``revision.csv``, ``manifest.json`` y, en ``vistas/``, DUPLICADOS y RESUMEN.
        ``golden.parquet`` no se escribe en esta ruta: ``dedupe()`` no produce el
        golden en memoria (quedan los golden por regimen en ``trabajo/``).
    """
    t0 = time.time()
    base = leer_base(ruta_archivo, hoja=hoja, columnas_texto=(col_nit,))
    df = exigir_columnas(base, {col_nit: "NIT", col_nombre: "RAZON_SOCIAL"}, "BASE")
    print(f"Base leida: {len(df):,} registros")

    resultado = dedupe(df, mode=modo, output_dir=str(Path(carpeta_salida) / "trabajo"))
    manifiesto = escribir_resultado(resultado, Path(carpeta_salida), nombre="deduplicacion")
    carpeta = manifiesto.carpeta
    print(f"Carpeta del estandar: {carpeta}")

    duplicados = tabla_duplicados(resultado.correlativa)
    m = resultado.metricas
    resumen = pd.DataFrame(
        {
            "indicador": [
                "registros_entrada",
                "grupos_unicos",
                "registros_en_duplicados",
                "registros_con_nit",
                "registros_sin_nit",
                "modo",
                "duracion_seg",
                "carpeta",
            ],
            "valor": [
                m["n_registros"],
                m["n_grupos"],
                len(duplicados),
                m["n_registros_con_nit"],
                m["n_registros_sin_nit"],
                modo,
                round(time.time() - t0, 1),
                str(carpeta),
            ],
        }
    )
    for ruta in exportar_vistas(
        {"DUPLICADOS": duplicados, "RESUMEN": resumen}, carpeta / "vistas", "deduplicacion"
    ):
        print(f"  Vista escrita: {ruta}")
    print()
    print(resultado.resumen())
    del base, df, resultado
    gc.collect()
    return carpeta

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CELDA B — CONFIGURAR Y EJECUTAR        (LA UNICA CELDA QUE USTED EDITA)
# ══════════════════════════════════════════════════════════════════════════
ARCHIVO = "mi_base.xlsx"  # @param {type:"string"}
HOJA = "Hoja1"  # @param {type:"string"}
COL_NIT = "NIT"  # @param {type:"string"}
COL_NOMBRE = "RAZON SOCIAL"  # @param {type:"string"}
MODO = "BALANCEADO"  # @param ["AGRESIVO", "BALANCEADO", "CONSERVADOR"]

CARPETA = deduplicar_una_base(RUTA_DATOS / ARCHIVO, HOJA, COL_NIT, COL_NOMBRE, MODO, RUTA_RESULTADOS)

# La carpeta se lee de vuelta con la libreria: verifica las huellas del
# manifiesto y devuelve el mismo ResultadoLinkage que produjo el motor.
res = leer_resultado(CARPETA)
print(res.resumen())
tabla_duplicados(res.correlativa).head(20)

### Como leer el resultado

`deduplicar_una_base` deja UNA carpeta con la forma del estandar de salida
(contrato 1.0, escrita por `escribir_resultado`):
`<RUTA_RESULTADOS>/<AAAA-MM-DD_HHMM>_deduplicacion/`

- **correlativa.parquet** (y `excel/correlativa.xlsx`): LA tabla — todos sus registros,
  cada uno con `ID_REGISTRO`, `ID_GRUPO`, `ID_ENTIDAD`, `METODO_UNION` y `CONFIANZA`.
  Dos filas con el mismo `ID_GRUPO` son la **misma empresa** segun el motor.
- **manifest.json**: version, parametros, huella de los insumos y conteos de la corrida.
- **diccionario.csv**: que significa cada columna (`leer_resultado(carpeta, alias="es")`
  renombra las columnas en español).
- **revision.csv**: pares por decidir (vacio en esta ruta).
- **vistas/**: lo que este notebook agrega — `DUPLICADOS` (solo los grupos con 2 o mas
  registros, lo accionable) y `RESUMEN` (conteos de la corrida).
- `golden.parquet` no se escribe aqui: `dedupe()` deja los golden por regimen en `trabajo/`.

`leer_resultado(carpeta)` verifica las huellas del manifiesto y devuelve el
`ResultadoLinkage` (`res.correlativa`, `res.manifiesto`, `res.metricas`).

**Sobre el modo**: `AGRESIVO` une mas (mayor cobertura, algun falso positivo posible),
`CONSERVADOR` une menos (maxima precision), `BALANCEADO` es el punto medio recomendado.

**Si algo falla**, el mensaje de error indica exactamente que corregir (ruta, hoja o
nombre de columna). Nada mas que ajustar: el motor calibra el resto solo.